<a href="https://colab.research.google.com/github/tsal4/data2000_labs/blob/main/exams/sp2026-midterm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA-2000 Midterm Exam


## Diabetes Readmission Prediction


For this exercise, we are going to use a dataset of hospitalized patient records diagnosed with diabetes, taken from [the University of California, Irvine ML Repository](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008).

The dataset represents ten years (1999-2008) of clinical care at 130 US hospitals and integrated delivery networks. Each row concerns hospital records of patients diagnosed with diabetes, who underwent laboratory, medications, and stayed up to 14 days. The goal is to determine the early readmission of the patient within 30 days of discharge. The problem is important for the following reasons. Despite high-quality evidence showing improved clinical outcomes for diabetic patients who receive various preventive and therapeutic interventions, many patients do not receive them. This can be partially attributed to arbitrary diabetes management in hospital environments, which fail to attend to glycemic control. Failure to provide proper diabetes care not only increases the managing costs for the hospitals (as the patients are readmitted) but also impacts the morbidity and mortality of the patients, who may face complications associated with diabetes.


## Grading Rubric

This midterm will be worth 15% of your total grade for this course. It will be graded out of 50 points, divided into 4 sections:

  - Data Prep: 10 points
    - 5 points will be awarded for the actual data cleaning (evaluating your Python code)
    - 5 points will be awarded for the text commentary narrating your choices and explaining your rationale for the data quality checks that you chose to use
  - Feature Engineering: 12 points
    - 2 points will be awarded by default, but may be subtracted from if there are substantial errors in your data prep that reduce the quality of your engineered features
    - 5 points will be awarded for the actual feature engineering (evaluating your Python code)
    - 5 points will be awarded for the text commentary narrating your choices and explaining your rationale
  - Model Building: 14 points
    - 4 points will be awarded by default, but may be subtracted from if there are substantial errors in your feature engineering that reduce the quality of your model
    - 5 points will be awarded for the actual model building (evaluating your Python code)
    - 5 points will be awarded for the text commentary narrating your choices and explaining your rationale
  - Model Validation/Evaluation: 14 points
    - 4 points will be awarded by default, but may be subtracted from if there are substantial errors in your model building that negatively impact the validity of your model
    - 5 points will be awarded for the actual model validation and evaluation (evaluating your Python code)
    - 5 points will be awarded for the text commentary narrating your choices and explaining your rationale

> **NOTE:** You will NOT be evaluated on whether you model actually makes accurate predictions or not

### IMPORTANT:

In the rubric, I include points for a "text narrative" accompanying your code. To be clear, I am asking for text cells in this notebook, interspersed with your actual code, that discuss (in one or more paragraphs of prose English text) the choices that you make and their rationale. I want these to appear alongside your code, and not as a single large summary at the end of the notebook. Likewise, code comments do not fulfill this requirement. The point here is for you to articulate, and for me to be able to understand, your thought process and the considerations that went into the modelling choices you made throughout this exam. Refer to the sample midterm solution on Github if you have questions about what I'm looking for, or ask me in class for additional clarification.

## Using Additional Resources

This is an open-resource exam. You may use any available resources as references. I will be available for any questions that you have during the exam.

Remember that all work must still be your own, and that this exam is governed by the Policy on Academic Honesty outlined in our course syllabus.

-----

## Importing the Data

First, let's download our dataset and take a look at what it contains:

In [1]:
!pip install ucimlrepo

In [62]:
import gdown
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import ydf
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, RocCurveDisplay, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import plot_importance
import seaborn as sns
import re

In [28]:
from ucimlrepo import fetch_ucirepo

uci_dataset = fetch_ucirepo(id=296)
data = pd.concat(
    [
        uci_dataset.data.features,
        uci_dataset.data.targets
    ],
    axis=1
)

/usr/local/lib/python3.12/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [29]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [30]:
data.head(15)

,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
5,Caucasian,Male,[50-60),NaN,2,1,2,3,NaN,NaN,31,6,16,0,0,0,414,411,250,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30
6,Caucasian,Male,[60-70),NaN,3,1,2,4,NaN,NaN,70,1,21,0,0,0,414,411,V45,7,NaN,NaN,Steady,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
7,Caucasian,Male,[70-80),NaN,1,1,7,5,NaN,NaN,73,0,12,0,0,0,428,492,250,8,NaN,NaN,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30
8,Caucasian,Female,[80-90),NaN,2,1,4,13,NaN,NaN,68,2,28,0,0,0,398,427,38,8,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
9,Caucasian,Female,[90-100),NaN,3,3,4,12,NaN,InternalMedicine,33,3,18,0,0,0,434,198,486,8,NaN,NaN,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [31]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 48 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   race                      99493 non-null   object
 1   gender                    101766 non-null  object
 2   age                       101766 non-null  object
 3   weight                    3197 non-null    object
 4   admission_type_id         101766 non-null  int64 
 5   discharge_disposition_id  101766 non-null  int64 
 6   admission_source_id       101766 non-null  int64 
 7   time_in_hospital          101766 non-null  int64 
 8   payer_code                61510 non-null   object
 9   medical_specialty         51817 non-null   object
 10  num_lab_procedures        101766 non-null  int64 
 11  num_procedures            101766 non-null  int64 
 12  num_medications           101766 non-null  int64 
 13  number_outpatient         101766 non-null  int64 
 14  numb

## Data Prep & Cleaning

Perform any data quality checks and data cleaning that you believe is appropriate. Convert any categorical columns to numeric ones, if needed. Provide a narrative explanation of your choices to accompany any code.

**I started by using data.info() and looking on the website to find data descriptions, type, and % missing. All columns are either int or object. I will start by removing some columns with lots of missing data and/or seem to be irrelevant.
Weight,  max_glu_serum, A1Cresult are all at least 90%+ NaN. Also payer coder and medical speciality seem irrelevant to personal health and are both over 50% missing.**

In [32]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 48 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   race                      99493 non-null   object
 1   gender                    101766 non-null  object
 2   age                       101766 non-null  object
 3   weight                    3197 non-null    object
 4   admission_type_id         101766 non-null  int64 
 5   discharge_disposition_id  101766 non-null  int64 
 6   admission_source_id       101766 non-null  int64 
 7   time_in_hospital          101766 non-null  int64 
 8   payer_code                61510 non-null   object
 9   medical_specialty         51817 non-null   object
 10  num_lab_procedures        101766 non-null  int64 
 11  num_procedures            101766 non-null  int64 
 12  num_medications           101766 non-null  int64 
 13  number_outpatient         101766 non-null  int64 
 14  numb

In [33]:
data = data.drop(['weight', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult'], axis=1)

**Now I need to take care of NaN values. I will handle each column differently.**

**I will fill the race NaN with "Other." There are only a small number of race Nan values so I'm just throwing them in with the misc "Other" value.**

In [34]:
data['race'] = data['race'].fillna("Other")

**Some number_diagnoses values are 0-2 meaning the patient did not receive 3 diagnoses, meaning that diag_1, diag_2, and diag_3 columns could be NaN. I will replace all NaN values in these columns with 0.**

In [35]:
cols = ['diag_1', 'diag_2', 'diag_3']
data[cols] = data[cols].fillna('0')

**Next, I will try to find outliers.**

In [36]:
data.describe()

,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,2.024006,3.715642,5.754437,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,1.445403,5.280166,4.064081,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,8.000000,28.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


**I disregard the ID variables as those will soon be converted to categories. They do not measure anything and are more so used as codes.**

**I used this quantile code for each of the numeric variables. I just replaced the variable name instead of having multiple lines of the same code.**

In [37]:
for val in (0, 0.01, 0.1, 0.25, 0.75, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999, 0.9995, 0.9999, 1):
    print(f"{val}:\t {data['number_diagnoses'].quantile(val)}")

0:	 1.0
0.01:	 2.0
0.1:	 5.0
0.25:	 6.0
0.75:	 9.0
0.9:	 9.0
0.95:	 9.0
0.98:	 9.0
0.99:	 9.0
0.995:	 9.0
0.999:	 10.0
0.9995:	 15.0
0.9999:	 16.0
1:	 16.0


**These are all of the variables with outliers. I cut them all at appropriate percentile using my intuition.**

In [38]:
data = data.loc[data["num_lab_procedures"] <= 112, :]

In [39]:
data = data.loc[data["num_medications"] <= 69, :]

In [40]:
data = data.loc[data["number_outpatient"] <= 18, :]

In [41]:
data = data.loc[data["number_emergency"] <= 13, :]

In [42]:
data = data.loc[data["number_inpatient"] <= 12, :]

**The last thing to do is change the dtype of the objects to categorical so the model can work with them.**

In [43]:
for col, dtype in data.dtypes.items():
    if dtype == "O":
        data[col] = data[col].astype("category")

In [44]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 101606 entries, 0 to 101765
Data columns (total 43 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   race                      101606 non-null  category
 1   gender                    101606 non-null  category
 2   age                       101606 non-null  category
 3   admission_type_id         101606 non-null  int64   
 4   discharge_disposition_id  101606 non-null  int64   
 5   admission_source_id       101606 non-null  int64   
 6   time_in_hospital          101606 non-null  int64   
 7   num_lab_procedures        101606 non-null  int64   
 8   num_procedures            101606 non-null  int64   
 9   num_medications           101606 non-null  int64   
 10  number_outpatient         101606 non-null  int64   
 11  number_emergency          101606 non-null  int64   
 12  number_inpatient          101606 non-null  int64   
 13  diag_1                    101606 n

**Now the data is clean and ready for feature engineering. Only 5 columns and about 100 rows were removed.**

## Feature Engineering

Develop any new feature(s) that you feel may be relevant to a model. Provide a narrative explanation of your choices to accompany any code.

**First, I will add a new column called total_visits which is number_outpatient + number_inpatient. This provides a general variable that shows how many a times a patient has visited the hospital. It is meant to provide a more general outlook.**

In [45]:
data["total_visits"] = data.loc[:, ["number_outpatient", "number_inpatient"]].sum(axis=1)

In [46]:
data["total_visits"].describe()

,total_visits
count,101606.000000
mean,0.983682
std,1.755143
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,22.000000


**I will do the same thing with num_lab_procedures and num_procedures to make total_procedures. I will add them into one variable to provide a more general outlook on how many procedures a patient went through. Both total_visits and total_procedures are meant to give a more general, first glance look at how a patient is doing outside of the hospital (total_visits) and how they are doing inside the hospital (total_procedures).**

In [47]:
data["total_procedures"] = data.loc[:, ["num_lab_procedures", "num_procedures"]].sum(axis=1)

In [48]:
data["total_procedures"].describe()

,total_procedures
count,101606.000000
mean,44.424670
std,19.826519
min,1.000000
25%,33.000000
50%,45.000000
75%,58.000000
max,117.000000


**Now I will split the data into training and testing sets.**

In [50]:
label = "readmitted"

le = LabelEncoder()
data_labels = data[label]
data_labels = le.fit_transform(data_labels)

X_train, X_test, y_train, y_test = train_test_split(
    data.drop([label], inplace=False, axis=1),
    data_labels,
    train_size=0.7
)

## Model Building

Build a random forest to predict a patient's likelihood to be readmitted within 30 days based on the attributes that you defined in the prior steps.

Provide a narrative explanation of your choices to accompany any code.

**I am using the standard classifier model that we used in class and I am not changing any of the parameters.**

In [54]:
clf = xgb.XGBClassifier(
    tree_method='hist',
    enable_categorical=True,
    objective='multi:softmax')
clf.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

**I tried using a graph to display the tree and show where the model made splits, but the model lists out every single individual value for the categorical variables. I need to keep them as categorical because the numbers are not measuring anything, they are instead used as codes that represent things like diagnoses and reason for admission to the hospital.**

**Sorry it looks weird I really tried to fix condense it and I even used AI but I couldn't figure it out.**

In [64]:
graph = xgb.to_graphviz(clf, num_trees=1)
graph

/usr/local/lib/python3.12/dist-packages/xgboost/plotting.py:268: FutureWarning: The `num_trees` parameter is deprecated, use `tree_idx` insetad. 
  warnings.warn(


## Model Evaluation

After training your model, evaluate its performance. What metric(s) did you choose to optimize on? Would you say that your model performed well or poorly? How did you evaluate its performance to arrive at that conclusion?

In [53]:
clf.score(X_test, y_test)

#confusion matrix
#feature importance
#XGBoost
plot_importance(clf, max_num_features=10, importance_type='gain')
plt.show()

NameError: name 'model' is not defined

-----

# Midterm Submission

To submit this exam, in Canvas navigate to DATA-2000-51 > Assignments > Midterm Exam. Upload a saved copy of this notebook as your sumission.